In [2]:
import mediapipe as mp
import cv2
from dollarpy import Recognizer, Template, Point

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic

templates = []

In [3]:
def getPoints(videoURL, label):
    cap = cv2.VideoCapture(videoURL)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        points = []
        stroke_id = 1

        while cap.isOpened():
            ret, frame = cap.read()

            if ret == True:
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image.flags.writeable = False

                results = holistic.process(image)

                image.flags.writeable = True
                image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

                if results.right_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        image,
                        results.right_hand_landmarks,
                        mp_holistic.HAND_CONNECTIONS
                    )

                    index_tip = results.right_hand_landmarks.landmark[8]

                    h, w, _ = image.shape
                    x = int(index_tip.x * w)
                    y = int(index_tip.y * h)

                    points.append(Point(x, y, stroke_id))
                    cv2.circle(image, (x, y), 8, (0, 255, 0), -1)

                cv2.putText(
                    image, label, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2
                )

                cv2.imshow("Gesture Video", image)

                if cv2.waitKey(10) & 0xFF == ord("q"):
                    break
            else:
                break

    cap.release()
    cv2.destroyAllWindows()
    return points

In [4]:
vid = " circle/circle_1.mp4"
points = getPoints(vid, "circle")

print(points[:10])
print("Total points:", len(points))

[(389, 460), stroke 1, (387, 459), stroke 1, (391, 460), stroke 1, (394, 462), stroke 1, (401, 465), stroke 1, (412, 469), stroke 1, (426, 475), stroke 1, (445, 480), stroke 1, (465, 483), stroke 1, (489, 490), stroke 1]
Total points: 69


In [5]:
templates = []

vid = " circle/circle_1.mp4"
points = getPoints(vid, "circle")
temp = Template("circle", points)
templates.append(temp)

vid = " circle/circle_2.mp4"
points = getPoints(vid, "circle")
temp = Template("circle", points)
templates.append(temp)

vid = " slide_left/slide_left_1.mp4"
points = getPoints(vid, "slide_left")
temp = Template("slide_left", points)
templates.append(temp)

vid = " slide_left/slide_left_2.mp4"
points = getPoints(vid, "slide_left")
temp = Template("slide_left", points)
templates.append(temp)

vid = " slide_right/slide_right_1.mp4"
points = getPoints(vid, "slide_right")
temp = Template("slide_right", points)
templates.append(temp)

vid = " slide_right/slide_right_2.mp4"
points = getPoints(vid, "slide_right")
temp = Template("slide_right", points)
templates.append(temp)

vid = " fist/fist_1.mp4"
points = getPoints(vid, "fist")
temp = Template("fist", points)
templates.append(temp)

vid = " fist/fist_2.mp4"
points = getPoints(vid, "fist")
temp = Template("fist", points)
templates.append(temp)

print("Templates loaded:", len(templates))

Templates loaded: 8


In [6]:
recognizer = Recognizer(templates)

vid = " circle/circle_3.mp4"
points = getPoints(vid, "circle test")
result = recognizer.recognize(points)

print("Predicted:", result[0])
print("Score:", result[1])

Predicted: circle
Score: 0.48821593663047347


In [7]:
recognizer = Recognizer(templates)

vid = " slide_left/slide_left_3.mp4"
points = getPoints(vid, "slide_left test")
result = recognizer.recognize(points)

print("Predicted:", result[0])
print("Score:", result[1])

Predicted: slide_left
Score: 0.7235851137310343


In [8]:
recognizer = Recognizer(templates)

vid = " slide_right/slide_right_3.mp4"
points = getPoints(vid, "slide_right test")
result = recognizer.recognize(points)

print("Predicted:", result[0])
print("Score:", result[1])

Predicted: slide_right
Score: 0.013606349840353027


In [9]:
recognizer = Recognizer(templates)

vid = " fist/fist_3.mp4"
points = getPoints(vid, "fist test")
result = recognizer.recognize(points)

print("Predicted:", result[0])
print("Score:", result[1])

Predicted: fist
Score: 0.515436745182371


In [10]:
label = result[0]

if label == "slide_right":
    print("Move category right")
elif label == "slide_left":
    print("Move category left")
elif label == "circle":
    print("Change current item")
elif label == "fist":
    print("Check answer")

Check answer


In [11]:
import mediapipe as mp
import cv2
import socket
import time
from dollarpy import Recognizer, Template, Point

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

def getPoints(videoURL, label):
    cap = cv2.VideoCapture(videoURL)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        points = []
        stroke_id = 1

        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            results = holistic.process(image)

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS
                )

                index_tip = results.right_hand_landmarks.landmark[8]

                h, w, _ = image.shape
                x = int(index_tip.x * w)
                y = int(index_tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)

        cap.release()
        cv2.destroyAllWindows()
        return points

templates = []

def add_template(label, video_path):
    points = getPoints(video_path, label)
    print(label, video_path, "points =", len(points))

    if len(points) > 0:
        templates.append(Template(label, points))
    else:
        print("Skipped empty template:", video_path)

add_template("circle", " circle/circle_1.mp4")
add_template("circle", " circle/circle_2.mp4")
add_template("slide_left", " slide_left/slide_left_1.mp4")
add_template("slide_left", " slide_left/slide_left_2.mp4")
add_template("slide_right", " slide_right/slide_right_1.mp4")
add_template("slide_right", " slide_right/slide_right_2.mp4")
add_template("fist", " fist/fist_1.mp4")
add_template("fist", " fist/fist_2.mp4")

print("Templates loaded:", len(templates))

if len(templates) == 0:
    raise Exception("No valid templates were loaded.")

recognizer = Recognizer(templates)

soc = socket.socket()
soc.bind(("localhost", 5001))
soc.listen(1)

print("Waiting for C# game to connect...")
conn, addr = soc.accept()
print("C# connected:", addr)

last_label = ""
last_time = 0

def send_gesture(label):
    global last_label, last_time

    now = time.time()

    if label == last_label and now - last_time < 1.0:
        return

    if label == "slide_right":
        msg = "GESTURE:RIGHT"
    elif label == "slide_left":
        msg = "GESTURE:LEFT"
    elif label == "circle":
        msg = "GESTURE:CIRCLE"
    elif label == "fist":
        msg = "GESTURE:FIST"
    else:
        return

    conn.send(msg.encode("utf-8"))
    print("Sent:", msg)

    last_label = label
    last_time = now

def recognize_live_gesture():
    cap = cv2.VideoCapture(0)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        points = []
        stroke_id = 1
        tracking = False

        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            results = holistic.process(image)

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS
                )

                index_tip = results.right_hand_landmarks.landmark[8]

                h, w, _ = image.shape
                x = int(index_tip.x * w)
                y = int(index_tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)
                tracking = True

            else:
                if tracking and len(points) > 10 and len(templates) > 0:
                    try:
                        result = recognizer.recognize(points)

                        if result is not None:
                            label = result[0]
                            score = result[1]

                            print("Predicted:", label)
                            print("Score:", score)

                            send_gesture(label)
                    except Exception as e:
                        print("Recognition error:", e)

                points = []
                tracking = False

            cv2.imshow("Live Gesture Recognition", image)

            key = cv2.waitKey(1) & 0xFF
            if key == 27:
                break

        cap.release()
        cv2.destroyAllWindows()

recognize_live_gesture()

conn.close()
soc.close()

circle  circle/circle_1.mp4 points = 69
circle  circle/circle_2.mp4 points = 105
slide_left  slide_left/slide_left_1.mp4 points = 40
slide_left  slide_left/slide_left_2.mp4 points = 23
slide_right  slide_right/slide_right_1.mp4 points = 11
slide_right  slide_right/slide_right_2.mp4 points = 15
fist  fist/fist_1.mp4 points = 58
fist  fist/fist_2.mp4 points = 51
Templates loaded: 8
Waiting for C# game to connect...
C# connected: ('127.0.0.1', 61075)
